# 04 · Multiple Comparisons & FDR Control

## What this notebook covers
When you test many hypotheses simultaneously — comparing 8 models pairwise = 28 tests — the probability of at least one false positive explodes. This notebook demonstrates the inflation and shows how to control it with Bonferroni (conservative) and Benjamini-Hochberg FDR (more powerful).

## The inflation problem
With k independent tests at α=0.05, the family-wise error rate (FWER) is:

    P(at least one false positive) = 1 − (1 − α)^k

For k=28: 1 − 0.95²⁸ ≈ 0.76. You'd expect ~1.4 false discoveries even if all nulls are true.

## Correction methods
| Method | Controls | Power | Use when |
|--------|----------|-------|---------|
| **Bonferroni** | FWER | Low | You need very strong control, small k |
| **Holm-Bonferroni** | FWER | Moderate | Step-down improvement over Bonferroni |
| **Benjamini-Hochberg** | FDR | High | Many tests, some true effects expected |

FDR (false discovery rate) is the expected fraction of rejected nulls that are actually false. BH is the standard in large-scale ML comparison tables.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                               ExtraTreesClassifier, AdaBoostClassifier)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

X, y = make_classification(n_samples=2000, n_features=30, n_informative=15, random_state=0)

models = {
    "GBM"  : GradientBoostingClassifier(n_estimators=100, random_state=0),
    "RF"   : RandomForestClassifier(n_estimators=100, random_state=0),
    "ET"   : ExtraTreesClassifier(n_estimators=100, random_state=0),
    "Ada"  : AdaBoostClassifier(n_estimators=100, random_state=0),
    "LR"   : LogisticRegression(max_iter=1000, random_state=0),
    "Ridge": RidgeClassifier(),
    "SVM"  : SVC(probability=True, random_state=0),
    "KNN"  : KNeighborsClassifier(n_neighbors=15),
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
fold_scores = {}
for name, clf in models.items():
    fold_scores[name] = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")

means = {n: s.mean() for n, s in fold_scores.items()}
print("Mean 10-fold AUC:")
for n, v in sorted(means.items(), key=lambda x: -x[1]):
    print(f"  {n:6s}: {v:.4f}")

In [ ]:
# All pairwise Wilcoxon tests
pairs, raw_p = [], []
model_names = list(fold_scores.keys())
for a, b in itertools.combinations(model_names, 2):
    _, p = stats.wilcoxon(fold_scores[a], fold_scores[b])
    pairs.append((a, b))
    raw_p.append(p)

raw_p = np.array(raw_p)

# Corrections
_, p_bonf, _, _ = multipletests(raw_p, method="bonferroni")
_, p_bh,   _, _ = multipletests(raw_p, method="fdr_bh")

results = pd.DataFrame({
    "Pair"   : [f"{a} vs {b}" for a, b in pairs],
    "raw p"  : raw_p.round(4),
    "Bonf p" : p_bonf.round(4),
    "BH p"   : p_bh.round(4),
    "Sig (raw)"  : raw_p < 0.05,
    "Sig (Bonf)" : p_bonf < 0.05,
    "Sig (BH)"   : p_bh < 0.05,
})
print(results.to_string(index=False))
print(f"\nDiscoveries — raw: {(raw_p<0.05).sum()}  Bonf: {(p_bonf<0.05).sum()}  BH: {(p_bh<0.05).sum()}")

In [ ]:
# BH threshold visualisation: rank vs sorted p-values
sorted_p = np.sort(raw_p)
m = len(sorted_p)
alpha = 0.05
ranks = np.arange(1, m+1)
bh_line = (ranks / m) * alpha

plt.figure(figsize=(9, 4))
plt.plot(ranks, sorted_p, "o", color="steelblue", label="Sorted p-values")
plt.plot(ranks, bh_line, "--", color="red", label=f"BH threshold (α={alpha})")
plt.axhline(alpha / m, color="orange", linestyle=":", label=f"Bonferroni threshold")
# Mark BH discoveries
bh_max_rank = np.max(np.where(sorted_p <= bh_line)[0]) + 1 if (sorted_p <= bh_line).any() else 0
if bh_max_rank > 0:
    plt.axvspan(0.5, bh_max_rank + 0.5, alpha=0.1, color="green", label=f"BH discoveries (k={bh_max_rank})")
plt.xlabel("Rank")
plt.ylabel("p-value")
plt.title("Benjamini-Hochberg correction: rank vs sorted p-values")
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{base}/04_multiple_comparisons.png", dpi=120)
plt.show()